In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config
import csv, io
from urllib.parse import urlparse

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

In [0]:
out_path = data_paths['dq']

res = []
for table_name in source_etl_validation_tables.keys(): 
    print(f'Validating {table_name}')

    df = spark.table(source_etl_validation_tables[table_name])
    res += validations.validate_table(
        spark=spark,
        tabletype="intermediate",
        tablename=table_name,
        config_validation=config_validation,
        df=df,
        stats_path=stats_etl_path,
        check_list=[
            validations.CompareColAggregatesPrior,
            validations.TestOutlierDays,
            validations.TestColNames,
            validations.TestDuplicates,
            # validations.TestControlTable,
        ],
        archive=False,
        throw_errors=False,
    )

In [0]:
keys = list(res[0].keys())
csv_buffer = io.StringIO()
writer = csv.DictWriter(csv_buffer, fieldnames=keys)
writer.writeheader()
writer.writerows(res)
csv_str = csv_buffer.getvalue()

# Expect out_path like: dbfs:/Volumes/<catalog>/<schema>/<volume>/reports/validation_report.csv
if not out_path.startswith("dbfs:/Volumes/"):
    raise ValueError(
        "out_path must be a Volume path like "
        "'dbfs:/Volumes/<catalog>/<schema>/<volume>/.../file.csv'"
    )

# Ensure parent directory exists
parent_dir = out_path.rsplit("/", 1)[0]
dbutils.fs.mkdirs(parent_dir)

# Overwrite the file in the Volume
dbutils.fs.put(out_path, csv_str, overwrite=True)

print(f"Wrote validation report to: {out_path}")